# Territorial.io PPO Training - Kaggle GPU
===

Run this notebook on Kaggle with GPU enabled (P100)

In [ ]:
!pip install kaggle -q

In [ ]:
"""
Territorial.io PPO Training - Kaggle GPU
====================================================
Author: AI Assistant
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import random
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    episodes = 5000
    max_steps_per_episode = 2000
    batch_size = 64
    ppo_epochs = 10
    ppo_clip_eps = 0.2
    gamma = 0.99
    gae_lambda = 0.95
    hidden_size = 256
    learning_rate = 3e-4
    value_loss_coef = 0.5
    entropy_coef = 0.01
    max_grad_norm = 0.5
    
    map_size = 20
    initial_balance = 100
    opening_phase_seconds = 107
    max_balance_cap = 5000
    defender_advantage = 2.0
    
    reward_win = 100.0
    reward_loss = -50.0
    reward_territory_gain = 1.0
    
    log_interval = 10
    model_save_path = "/kaggle/working/models"

In [ ]:
# ============================================================================
# GAME ENVIRONMENT
# ============================================================================

class Tile:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.owner_id = None
        self.balance = 0
        self.is_ocean = False


class Player:
    def __init__(self, player_id, is_bot=True):
        self.id = player_id
        self.is_bot = is_bot
        self.balance = Config.initial_balance
        self.territory = []
        self.alive = True
        self.is_eliminated = False


class GameEnv:
    def __init__(self, num_bots=19):
        self.num_bots = num_bots
        self.total_players = num_bots + 1
        self.reset()
    
    def reset(self):
        self.map = [[Tile(x, y) for y in range(Config.map_size)] 
                    for x in range(Config.map_size)]
        
        # Add oceans
        for x in range(Config.map_size):
            self.map[x][0].is_ocean = True
            self.map[x][Config.map_size-1].is_ocean = True
        for y in range(Config.map_size):
            self.map[0][y].is_ocean = True
            self.map[Config.map_size-1][y].is_ocean = True
        
        self.players = []
        self.ai_player = Player(0, is_bot=False)
        self.players.append(self.ai_player)
        
        for i in range(1, self.total_players):
            self.players.append(Player(i, is_bot=True))
        
        # Spawn
        available = [t for row in self.map for t in row if not t.is_ocean]
        random.shuffle(available)
        
        for p in self.players:
            if available:
                t = available.pop(0)
                t.owner_id = p.id
                p.territory.append(t)
        
        self.time_elapsed = 0.0
        self.game_over = False
        self.winner = None
        self.max_territory = 0
        
        return self._get_state()
    
    def step(self, action):
        target_id, pct = action
        
        # AI action
        if target_id == -1:
            self._expand(self.ai_player, pct)
        else:
            self._attack(self.ai_player, target_id, pct)
        
        # Bot actions
        for bot in self.players[1:]:
            if bot.alive:
                tgt, p = self._bot_decision(bot)
                if tgt is None:
                    self._expand(bot, p)
                else:
                    self._attack(bot, tgt, p)
        
        # Interest
        self.time_elapsed += 1
        if int(self.time_elapsed) % 5 == 0:
            self._apply_interest()
        
        # Check elimination
        for p in self.players:
            if len(p.territory) == 0 and p.balance < 10:
                p.is_eliminated = True
                p.alive = False
        
        # Check game over
        alive = [p for p in self.players if p.alive and not p.is_eliminated]
        if len(alive) == 1:
            self.game_over = True
            self.winner = alive[0].id
        
        if self.ai_player.is_eliminated:
            self.game_over = True
        
        reward = self._get_reward()
        
        return self._get_state(), reward, self.game_over, {}
    
    def _expand(self, player, pct):
        if not player.alive:
            return
        
        troops = int(player.balance * pct / 100)
        if troops < 1:
            return
        
        player.balance -= troops
        
        # Find empty neighbors
        empty = []
        for t in player.territory:
            for dx, dy in [(0,1),(0,-1),(1,0),(-1,0)]:
                nx, ny = t.x + dx, t.y + dy
                if 0 <= nx < Config.map_size and 0 <= ny < Config.map_size:
                    adj = self.map[nx][ny]
                    if adj.owner_id is None and not adj.is_ocean:
                        empty.append(adj)
        
        if empty:
            for _ in range(min(3, len(empty))):
                if empty:
                    t = empty.pop(0)
                    t.owner_id = player.id
                    player.territory.append(t)
    
    def _attack(self, attacker, target_id, pct):
        if not attacker.alive:
            return
        
        target = self.players[target_id]
        if not target.alive:
            return
        
        troops = int(attacker.balance * pct / 100)
        if troops < 1:
            return
        
        attacker.balance -= troops
        
        # Battle (defender advantage 2:1)
        defender_loss = troops // 2
        
        if len(target.territory) > 0:
            for _ in range(min(defender_loss, len(target.territory))):
                if target.territory:
                    t = random.choice(target.territory)
                    t.owner_id = attacker.id
                    attacker.territory.append(t)
                    target.territory.remove(t)
    
    def _bot_decision(self, bot):
        enemies = []
        for t in bot.territory:
            for dx, dy in [(0,1),(0,-1),(1,0),(-1,0)]:
                nx, ny = t.x + dx, t.y + dy
                if 0 <= nx < Config.map_size and 0 <= ny < Config.map_size:
                    adj = self.map[nx][ny]
                    if adj.owner_id is not None and adj.owner_id != bot.id:
                        e = self.players[adj.owner_id]
                        if e.alive and not e.is_eliminated:
                            enemies.append(e.id)
        
        if not enemies:
            return None, 25
        
        target = random.choice(enemies)
        
        if self.time_elapsed < 107:
            return target, 20
        else:
            return target, 30
    
    def _apply_interest(self):
        rate = max(0, 0.5 * (1 - self.time_elapsed / 120))
        for p in self.players:
            if p.alive and not p.is_eliminated:
                bonus = len(p.territory) * 10
                max_cap = min(Config.max_balance_cap, 100 + bonus)
                p.balance += int(min(p.balance, max_cap) * rate)
    
    def _get_reward(self):
        r = 0
        curr = len(self.ai_player.territory)
        r += (curr - self.max_territory) * Config.reward_territory_gain
        self.max_territory = max(self.max_territory, curr)
        r += self.ai_player.balance * 0.001
        r += 0.01
        
        if self.game_over:
            if self.winner == 0:
                r += Config.reward_win
            else:
                r += Config.reward_loss
        
        return r
    
    def _get_state(self):
        s = []
        s.append(min(self.ai_player.balance / Config.max_balance_cap, 1.0))
        s.append(len(self.ai_player.territory) / 400)
        s.append(self.time_elapsed / 300)
        
        alive = sum(1 for p in self.players if p.alive and not p.is_eliminated)
        s.append(alive / self.total_players)
        
        while len(s) < 50:
            s.append(0.0)
        
        return np.array(s[:50], dtype=np.float32)
    
    def get_valid_actions(self):
        actions = [(-1, p) for p in range(10, 100, 10)]
        
        for bot in self.players[1:]:
            if bot.alive and not bot.is_eliminated:
                for p in range(10, 100, 10):
                    actions.append((bot.id, p))
        
        return actions

In [ ]:
# ============================================================================
# PPO AGENT
# ============================================================================

class ActorCritic(nn.Module):
    def __init__(self, state_dim=50, action_dim=100, hidden_dim=256):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        self.actor = nn.Linear(hidden_dim, action_dim)
        self.critic = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        h = self.net(x)
        return self.actor(h), self.critic(h)


class PPOAgent:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Device: {self.device}")
        
        self.policy = ActorCritic().to(self.device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=Config.learning_rate)
        
        self.buffer = []
    
    def select_action(self, state, valid_actions):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            logits, value = self.policy(state_t)
        
        if not valid_actions:
            return (-1, 20), value.item()
        
        probs = torch.softmax(logits[0][:len(valid_actions)], dim=0)
        idx = torch.multinomial(probs, 1).item()
        action = valid_actions[idx]
        
        return action, value.item()
    
    def update(self, batch):
        states, actions, rewards, old_log_probs, values = batch
        
        # FIXED: Convert to numpy array first
        states = torch.FloatTensor(np.array(states)).to(self.device)
        actions = torch.tensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        
        # Returns
        returns = rewards
        advantages = returns - torch.tensor(values).to(self.device)
        
        logits, values_new = self.policy(states)
        
        # PPO loss
        log_probs = torch.log_softmax(logits, dim=1)
        action_log_probs = log_probs.gather(1, actions.unsqueeze(1)).squeeze()
        
        ratio = torch.exp(action_log_probs - torch.tensor(old_log_probs).to(self.device))
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1-0.2, 1+0.2) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()
        
        value_loss = nn.functional.mse_loss(values_new.squeeze(), returns)
        
        loss = policy_loss + 0.5 * value_loss - 0.01 * (logits * torch.softmax(logits, dim=1)).sum()
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
    
    def save(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(self.policy.state_dict(), path)

In [ ]:
# ============================================================================
# TRAINING
# ============================================================================

def train(episodes=5000):
    print("=" * 50)
    print("Territorial.io PPO Training (Kaggle GPU)")
    print("=" * 50)
    
    env = GameEnv(num_bots=19)
    agent = PPOAgent()
    
    os.makedirs(Config.model_save_path, exist_ok=True)
    
    for ep in range(episodes):
        state = env.reset()
        ep_reward = 0
        
        for step in range(Config.max_steps_per_episode):
            valid = env.get_valid_actions()
            action, value = agent.select_action(state, valid)
            
            next_state, reward, done, _ = env.step(action)
            
            agent.buffer.append((state, action, reward, 0, value))
            
            if len(agent.buffer) >= Config.batch_size:
                batch = list(zip(*agent.buffer[-Config.batch_size:]))
                agent.update(batch)
            
            ep_reward += reward
            state = next_state
            
            if done:
                break
        
        if (ep + 1) % Config.log_interval == 0:
            print(f"Episode {ep+1}/{episodes} | Reward: {ep_reward:.1f}")
        
        if (ep + 1) % 100 == 0:
            agent.save(f"{Config.model_save_path}/model_ep{ep+1}.pt")
    
    agent.save(f"{Config.model_save_path}/final_model.pt")
    print("Training complete!")
    
    return agent


if __name__ == "__main__":
    agent = train(episodes=5000)